# 03 — Preprocessing and Feature Engineering

This notebook builds the data the models actually see, and demonstrates that the
three leakage controls the project claims are real rather than asserted:

1. **No target leakage** — `id`, `attack_cat` and `label` never enter the matrix.
2. **No preprocessing leakage** — scalers and encoders are fitted inside the
   training fold only.
3. **No duplicate leakage** — identical feature vectors never span train and test.

In [1]:
import sys, warnings
from pathlib import Path

# Make the repository root importable no matter where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=FutureWarning)

from src import config
print(f"Repository root : {ROOT}")
print(f"Random seed     : {config.RANDOM_STATE}")

Repository root : E:\AIM\AIM AI ML Capstone\AI_Capstone_Network_Intrusion_Detection
Random seed     : 42


## 1. Corpus preparation

Deduplication happens **before** any split. Doing it afterwards would leave
identical vectors straddling train and test — precisely the problem it exists to
prevent.

The cell below profiles the pooled corpus, which is how the duplication problem
is best *seen*. The primary protocol in section 2 does not pool: it keeps the
authors' partition and deduplicates each side separately.

In [2]:
from src import preprocessing

corpus, report = preprocessing.prepare_corpus(verbose=True)

print()
for key in ("rows_loaded", "duplicate_rows", "duplicate_fraction",
            "conflicting_feature_vectors", "rows_final", "missing_values"):
    print(f"  {key:<30} {report[key]}")

[preprocessing] loaded 257,673 rows
[preprocessing] duplicate feature vectors: 103,989 (40.4%)
[preprocessing] contradictory feature vectors: 414
[preprocessing] corpus after dedup=True: 153,684 rows
[preprocessing] class balance: {0: 0.5562, 1: 0.4438}

  rows_loaded                    257673
  duplicate_rows                 103989
  duplicate_fraction             0.4035696405909816
  conflicting_feature_vectors    414
  rows_final                     153684
  missing_values                 0


## 2. The primary split protocol

In [3]:
# The PRIMARY protocol: the authors' published partition, each side
# deduplicated, with development/test overlap removed. The pooled random split
# above is the easier alternative, carried as the `pooled_random` ablation.
train, val, test, manifest = preprocessing.split_published(verbose=True)

print()
for key, value in manifest.items():
    print(f"  {key:<34} {value}")

distribution = pd.DataFrame({
    name: part["attack_cat"].value_counts(normalize=True)
    for name, part in (("train", train), ("val", val), ("test", test))
}).sort_values("train", ascending=False)
print("\nAttack-family proportions, development vs the published test file:")
display(distribution.style.format("{:.4f}"))

[preprocessing] published protocol: train 80,832 / val 20,208 / test 52,644
[preprocessing]   duplicates removed - development 74,301, test 28,386
[preprocessing]   train/test overlap removed: 1,302
[preprocessing]   attack rate - train 0.4872, test 0.3606

  protocol                           published_partition
  published_train_rows               175341
  published_test_rows                82332
  development_conflicting_signatures 229
  development_duplicates_removed     74301
  test_conflicting_signatures        6
  test_duplicates_removed            28386
  test_overlap_removed               1302
  train_rows                         80832
  validation_rows                    20208
  test_rows                          52644
  train_attack_rate                  0.4872
  validation_attack_rate             0.4872
  test_attack_rate                   0.3606
  random_state                       42

Attack-family proportions, development vs the published test file:


,train,val,test
attack_cat,,,
Normal,0.5128,0.5128,0.6394
Exploits,0.1853,0.1853,0.1346
Fuzzers,0.1474,0.1474,0.0823
Reconnaissance,0.0636,0.0635,0.0422
Generic,0.0388,0.0388,0.0642
DoS,0.0297,0.0297,0.0225
Shellcode,0.0108,0.0108,0.0069
Analysis,0.0054,0.0053,0.0056
Backdoor,0.0051,0.0051,0.0013


Read the last column against the first two. The development splits match each
other closely, because validation is a stratified draw from the same file. The
**test column does not** — it is a different capture, and that mismatch is the
distribution shift the primary protocol exists to expose. A pooled random split
would have eliminated it by construction.

In [4]:
predictors = [c for c in train.columns
              if c not in (*config.LEAKAGE_COLUMNS, "partition")]

def signatures(frame):
    return set(map(tuple, frame[predictors].itertuples(index=False, name=None)))

s_train, s_val, s_test = signatures(train), signatures(val), signatures(test)
print("Shared feature vectors between splits (all must be zero):")
print(f"  train n val   {len(s_train & s_val)}")
print(f"  train n test  {len(s_train & s_test)}")
print(f"  val   n test  {len(s_val & s_test)}")
assert not (s_train & s_val) and not (s_train & s_test) and not (s_val & s_test)
print("\nNo feature vector appears in more than one split.")

Shared feature vectors between splits (all must be zero):
  train n val   0
  train n test  0
  val   n test  0

No feature vector appears in more than one split.


Stratification of the development split is on **`attack_cat`**, not on `label`.
Because `label` is a deterministic function of `attack_cat`, stratifying on the
family stratifies the binary target as a side effect — while additionally
guaranteeing that rare families appear in both train and validation. The binary
label alone would not ensure that.

The zero-overlap result above is the one that matters: it means the test metrics
measure generalisation rather than memorisation. Note that deduplicating each
partition does **not** give this for free — a record present in both published
files survives both passes, which is what the overlap-removal step catches.

## 3. Leakage enforcement

In [5]:
X_train, y_train = preprocessing.split_xy(train)
X_val, y_val = preprocessing.split_xy(val)
X_test, y_test = preprocessing.split_xy(test)

print(f"Predictors: {X_train.shape[1]}    Target: '{y_train.name}'\n")
for forbidden in (*config.LEAKAGE_COLUMNS, "partition"):
    status = "PRESENT — LEAK!" if forbidden in X_train.columns else "absent (correct)"
    print(f"  {forbidden:<12} {status}")

Predictors: 42    Target: 'label'

  id           absent (correct)
  attack_cat   absent (correct)
  label        absent (correct)
  partition    absent (correct)


## 4. Feature engineering

In [6]:
from src.features import ENGINEERED_FEATURE_DOCS, add_engineered_features

print(f"{len(ENGINEERED_FEATURE_DOCS)} engineered features:\n")
for name, rationale in ENGINEERED_FEATURE_DOCS.items():
    print(f"  {name}")
    print(f"      {rationale[:110]}...")

15 engineered features:

  flow_bytes_total
      Total bytes carried in both directions. Separates bulk transfers and data exfiltration from the tiny probe flo...
  flow_pkts_total
      Total packets in both directions. A volume counterpart to byte count that is insensitive to payload size, so i...
  bytes_per_packet
      Mean payload size across the whole flow. Floods and scans use minimal packets; exploit and shellcode deliverie...
  src_byte_ratio
      Share of flow bytes sent by the source, in [0, 1]. Values near 1.0 mean the source talked and the destination ...
  src_pkt_ratio
      Packet-count analogue of src_byte_ratio, in [0, 1]. Robust when payload sizes vary, e.g. reconnaissance that p...
  load_log_ratio
      log1p(source bits/s) - log1p(destination bits/s). A signed, scale-stable measure of throughput asymmetry: stro...
  jit_log_ratio
      log1p(source jitter) - log1p(destination jitter). Machine-generated attack traffic is often unnaturally regula...
  is_one_way


In [7]:
sample = X_train.head(6)
engineered = add_engineered_features(sample)

display(Markdown("**Raw flow measurements**"))
display(sample[["proto", "service", "state", "dur", "sbytes", "dbytes",
                "spkts", "dpkts", "synack", "ackdat"]])

display(Markdown("**Engineered from them**"))
display(engineered[list(ENGINEERED_FEATURE_DOCS)].round(4))

**Raw flow measurements**

,proto,service,state,dur,sbytes,dbytes,spkts,dpkts,synack,ackdat
0,tcp,-,fin,0.027304,3182,38972,52,52,0.000566,0.000118
1,tcp,ftp-data,fin,0.002721,320,1888,6,8,0.000450,0.000117
2,tcp,http,fin,0.565023,892,15720,10,18,0.047283,0.072832
3,tcp,http,fin,0.667744,806,1114,10,8,0.099370,0.112081
4,udp,dns,con,0.001057,146,178,2,2,0.000000,0.000000
5,tcp,-,fin,1.181850,13454,548216,234,438,0.000461,0.000119


**Engineered from them**

,flow_bytes_total,flow_pkts_total,bytes_per_packet,src_byte_ratio,src_pkt_ratio,load_log_ratio,jit_log_ratio,is_one_way,tcp_handshake_complete,tcp_seq_exchanged,both_win_advertised,is_zero_duration,service_unknown,src_loss_rate,dst_loss_rate
0,42154.0,104.0,405.3269,0.0755,0.5000,-2.5053,0.0136,0,1,1,1,0,1,0.1346,0.3846
1,2208.0,14.0,157.7143,0.1449,0.4286,-1.8225,2.9206,0,1,1,1,0,0,0.1667,0.2500
2,16612.0,28.0,593.2857,0.0537,0.3571,-2.9171,0.1983,0,1,1,1,0,0,0.2000,0.3889
3,1920.0,18.0,106.6667,0.4198,0.5556,-0.2949,3.3316,0,1,1,1,0,0,0.2000,0.2500
4,324.0,4.0,81.0000,0.4506,0.5000,-0.1982,0.0000,0,0,0,0,0,0,0.0000,0.0000
5,561670.0,672.0,835.8185,0.0240,0.3482,-3.7093,0.3153,0,1,1,1,0,1,0.0897,0.4498


### Why these are row-wise, and why that matters twice

Every engineered feature is a function of **one flow record**. None uses a
statistic estimated from the data — no target encoding, no dataset-level means,
no group aggregates. That property delivers two things at once:

1. **It cannot leak.** A feature that never sees the label cannot leak the
   target, and one that never sees another row cannot leak across the split.
2. **It is deployable.** Each can be computed by a sensor on a single flow at
   inference time, which is how an IDS must operate.

In [8]:
# Numerical safety on the degenerate flows that make up nearly half the dataset.
degenerate = X_train[(X_train["dpkts"] == 0) & (X_train["dur"] == 0)].head(3)
print(f"Flows with zero response AND zero duration in the training split: "
      f"{((X_train['dpkts'] == 0) & (X_train['dur'] == 0)).sum():,}\n")

if len(degenerate):
    out = add_engineered_features(degenerate)[list(ENGINEERED_FEATURE_DOCS)]
    display(out.round(4))
    print(f"\nAll finite: {np.isfinite(out.to_numpy(float)).all()}")

Flows with zero response AND zero duration in the training split: 403



,flow_bytes_total,flow_pkts_total,bytes_per_packet,src_byte_ratio,src_pkt_ratio,load_log_ratio,jit_log_ratio,is_one_way,tcp_handshake_complete,tcp_seq_exchanged,both_win_advertised,is_zero_duration,service_unknown,src_loss_rate,dst_loss_rate
10,46.0,1.0,46.0,1.0,1.0,0.0,2.4811,1,0,0,0,1,1,0.0,0.0
71,46.0,1.0,46.0,1.0,1.0,0.0,2.8029,1,0,0,0,1,1,0.0,0.0
78,46.0,1.0,46.0,1.0,1.0,0.0,0.0000,1,0,0,0,1,1,0.0,0.0



All finite: True


46.7% of flows have `dpkts == 0`, so an unguarded division would corrupt a large
fraction of the matrix rather than a rare edge case. Every ratio is formed as
`a / (a + b)` — bounded to [0, 1] — rather than `a / b`, every denominator is
clipped, and the module asserts on output that no non-finite value was produced.
`tests/test_features.py` verifies this on hand-built records.

## 5. The preprocessing pipeline

In [9]:
groups = preprocessing.feature_columns()
for name, columns in groups.items():
    print(f"{name:<12} ({len(columns):>2}) {', '.join(columns[:6])}"
          f"{' ...' if len(columns) > 6 else ''}")

log1p        (26) dur, spkts, dpkts, sbytes, dbytes, rate ...
linear       (21) sttl, dttl, swin, dwin, ct_srv_src, ct_state_ttl ...
flags        ( 7) is_sm_ips_ports, is_one_way, tcp_handshake_complete, tcp_seq_exchanged, both_win_advertised, is_zero_duration ...
categorical  ( 3) proto, service, state


In [10]:
pipeline = preprocessing.build_feature_pipeline()
pipeline.fit(X_train, y_train)

design = pipeline.transform(X_train)
names = preprocessing.transformed_feature_names(pipeline)

print(f"Design matrix : {design.shape}")
print(f"All finite    : {np.isfinite(design).all()}")
print(f"\nOne-hot columns produced:")
for prefix in ("proto_", "service_", "state_"):
    produced = [n for n in names if n.startswith(prefix)]
    print(f"  {prefix:<10} {len(produced):>2} — {', '.join(produced[:6])}"
          f"{' ...' if len(produced) > 6 else ''}")

Design matrix : (80832, 73)
All finite    : True

One-hot columns produced:
  proto_      5 — proto_arp, proto_tcp, proto_udp, proto_unas, proto_infrequent_sklearn
  service_   10 — service_unknown, service_-, service_dns, service_ftp, service_ftp-data, service_http ...
  state_      5 — state_con, state_fin, state_int, state_req, state_infrequent_sklearn


`proto` has 133 levels in the corpus but produces far fewer columns: levels seen
fewer than 200 times in training are folded into a single `infrequent` bucket.
That is both dimensionality control and a **generalisation choice** — the model
learns "this flow used an unusual protocol" rather than memorising which
protocol numbers were scanned in 2015. It also gives protocols never seen during
training a defined destination at inference time, instead of an exception.

In [11]:
# Proof that fitting used the training split ONLY.
scaler = pipeline.named_steps["preprocess"].named_transformers_["num"]
engineered_train = pipeline.named_steps["engineer"].transform(X_train)
expected = engineered_train[groups["linear"]].to_numpy(float).mean(axis=0)

print(f"Scaler means match the TRAINING split exactly: "
      f"{np.allclose(scaler.mean_, expected)}")

before = scaler.mean_.copy()
pipeline.transform(X_val)
print(f"Transforming the validation split left the fitted state unchanged: "
      f"{np.array_equal(scaler.mean_, before)}")

Scaler means match the TRAINING split exactly: True
Transforming the validation split left the fitted state unchanged: True


In [12]:
# An unseen protocol must be absorbed, not raise — the failure mode that takes a
# deployed detector offline the first time an unusual protocol crosses the wire.
novel = X_val.head(5).copy()
novel["proto"] = "protocol-invented-in-2030"
novel["service"] = "unheard-of-service"

result = pipeline.transform(novel)
print(f"Unseen categories handled: shape {result.shape}, all finite {np.isfinite(result).all()}")

Unseen categories handled: shape (5, 73), all finite True


## 6. Class imbalance

In [13]:
balance = y_train.value_counts(normalize=True).sort_index()
print(f"Training split: {balance[0]:.1%} benign / {balance[1]:.1%} attack")

from src.train import scale_pos_weight
print(f"scale_pos_weight (n_neg / n_pos) = {scale_pos_weight(y_train):.4f}")
print("\nImbalance is MILD and INVERTED (attacks are the larger class).")
print("class_weight / scale_pos_weight are included as TUNED hyper-parameters so")
print("the search decides empirically rather than by assumption.")

Training split: 51.3% benign / 48.7% attack


scale_pos_weight (n_neg / n_pos) = 1.0526

Imbalance is MILD and INVERTED (attacks are the larger class).
class_weight / scale_pos_weight are included as TUNED hyper-parameters so
the search decides empirically rather than by assumption.


### Why SMOTE is not used by default

SMOTE interpolates between neighbouring minority points. On this data that is
questionable: many features are Boolean flags or bounded counters, and a
synthetic flow with `tcp_handshake_complete = 0.43` does not correspond to any
packet sequence that could exist on a wire.

It is nevertheless **tested** rather than dismissed — `python -m src.train
--ablation smote` applies `SMOTENC` (which leaves nominal columns
un-interpolated) to the **training split only**, and the comparison is reported
in the Model Evaluation Report. The argument is settled by evidence, not
assertion.

## 7. Persist the splits

In [14]:
config.ensure_dirs()
for name, part in (("train", train), ("val", val), ("test", test)):
    path = config.PROCESSED_DIR / f"{name}.parquet"
    part.to_parquet(path, index=False)
    print(f"wrote data/processed/{name}.parquet  ({len(part):,} rows)")

wrote data/processed/train.parquet  (80,832 rows)
wrote data/processed/val.parquet  (20,208 rows)


wrote data/processed/test.parquet  (52,644 rows)


---

## Summary

| Control | Mechanism | Verified by |
|---|---|---|
| No target leakage | Single enforcement point in `split_xy` | Section 3; `tests/test_preprocessing.py` |
| No preprocessing leakage | Estimator shares a `Pipeline` with the transforms | Section 5; scaler-mean check |
| No duplicate leakage | Deduplicate before splitting | Section 2; zero-overlap check |
| Unseen categories safe | `min_frequency` + `handle_unknown` | Section 5 |
| Numerical safety | Guarded denominators, bounded ratios | Section 4; `tests/test_features.py` |
| Reproducible | `random_state=42`, splits materialised to parquet | Section 7 |

**Next:** [`04_model_training.ipynb`](04_model_training.ipynb)